# Làm sạch và biến đổi dữ liệu

## Mục tiêu
- Lập data-quality report trước khi sửa.
- Xử lý missing/type/duplicate có lý do.
- Encode và scale trong pipeline để không leakage.

In [ ]:
from pathlib import Path
import pandas as pd

repo_root = Path('.') if Path('data').exists() else Path('..')
df = pd.read_csv(repo_root / 'data/retail_customers.csv')
quality = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing': df.isna().sum(),
    'missing_rate': df.isna().mean(),
    'unique': df.nunique()
})
quality

## Làm sạch có thể audit
Không tự động điền mọi giá trị thiếu bằng mean. Với `income`, `not_available` là giá trị bị mã hoá sai; với `last_purchase_days`, thiếu có thể mang ý nghĩa business riêng và cần xác nhận với người cung cấp dữ liệu.

In [ ]:
clean = df.copy()
clean['income'] = pd.to_numeric(clean['income'], errors='coerce')
clean['age'] = clean['age'].fillna(clean['age'].median())
clean['last_purchase_days_missing'] = clean['last_purchase_days'].isna().astype(int)
clean['last_purchase_days'] = clean['last_purchase_days'].fillna(clean['last_purchase_days'].median())
clean = clean.drop_duplicates(subset='customer_id')
clean[['income', 'age', 'last_purchase_days']].describe()

## Encode và scale sau khi split
`StandardScaler` học mean/std từ data. Nếu fit nó trên toàn bộ data trước split, thông tin test đã rò vào train. `Pipeline` giữ thứ tự an toàn và tái lập được.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric = ['age', 'income', 'visits', 'last_purchase_days']
categorical = ['city', 'channel']
preprocessor = ColumnTransformer([
    ('numeric', StandardScaler(), numeric),
    ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical),
])
preprocessor

## Tự luyện
1. Viết bảng trước/sau cleaning gồm row count, duplicate count, missing count.
2. Thử thêm feature `spend_per_visit`; xử lý visits bằng 0 an toàn.
3. Với một dataset thật, ghi rõ mỗi quyết định impute dựa trên knowledge nào.

**Checkpoint:** Bạn giải thích được tại sao không fit scaler trên test set.